# 🧩 检索增强生成 (RAG)：从理论到工程实战

在大模型（LLM）的落地场景中，**RAG (Retrieval-Augmented Generation)** 是解决领域知识缺失和幻觉问题的常用手段。它的核心逻辑是：**“先搜索，再回答”**，将外部知识库作为模型的“开卷参考书”。

本节实验将深度解析 RAG 的 **四大支柱**，并采用 **工业级工程标准 (System + User Prompt)** 进行全流程构建。

---

## 🛠️ 1. 环境初始化：加载 RAG “三剑客”

一个高性能的 RAG 系统需要三种不同职责的模型协同工作：
1. **Embedding (嵌入)**: `bge-small-zh-v1.5` —— 负责将文本映射到高维空间。
2. **Reranker (重排)**: `bge-reranker-v2-m3` —— 负责对初步检索结果进行语义精排。
3. **Generation (生成)**: `Qwen2-0.5B-Instruct` —— 负责基于资料进行最终回答。

我们将使用“自愈式加载器”确保在 CPU 环境中可靠载入。

In [ ]:
import os
import torch
import torch.nn.functional as F
from pathlib import Path
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel, AutoModelForSequenceClassification
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# 1. 设置镜像站加速 (国内环境必备)
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

def load_rag_models():
    """加载 RAG 全流程所需的本地模型"""
    models_root = Path("models")
    # 对应三剑客：LLM, Embedding, Reranker
    llm_id, emb_id, rerank_id = "Qwen/Qwen2-0.5B-Instruct", "BAAI/bge-small-zh-v1.5", "BAAI/bge-reranker-v2-m3"
    
    def quick_load(repo_id, model_type="llm"):
        model_name = repo_id.split("/")[-1]
        local_path = models_root / model_name
        # snapshot_download 会物理校验文件完整性，若不完整则自愈补齐
        snapshot_download(repo_id=repo_id, local_dir=str(local_path), local_dir_use_symlinks=False, ignore_patterns=["*.msgpack", "*.h5", "*.ot"])
        
        tokenizer = AutoTokenizer.from_pretrained(str(local_path))
        if model_type == "llm":
            model = AutoModelForCausalLM.from_pretrained(str(local_path), device_map="cpu")
        elif model_type == "emb":
            model = AutoModel.from_pretrained(str(local_path)).to("cpu")
        elif model_type == "rerank":
            model = AutoModelForSequenceClassification.from_pretrained(str(local_path)).to("cpu")
        return model, tokenizer

    # 依次加载三个模型
    (llm, llm_tok), (emb, emb_tok), (rerank, rerank_tok) = quick_load(llm_id, "llm"), quick_load(emb_id, "emb"), quick_load(rerank_id, "rerank")
    # 修复 Padding Token 缺失问题
    if llm_tok.pad_token is None: llm_tok.pad_token = llm_tok.eos_token
    return (llm, llm_tok), (emb, emb_tok), (rerank, rerank_tok)

(llm, llm_tok), (emb, emb_tok), (rerank, rerank_tok) = load_rag_models()
print("\n✅ RAG 全链路模型已成功载入物理内存。")

## 🧱 2. 第一支柱：Chunking (分块策略)

### 🔎 为什么要进行文本切分？
大模型处理长文本的能力受到 **上下文窗口 (Context Window)** 的限制。如果不分块，长文档会超限；即使不超限，将过多无关信息塞入 Prompt 也会产生“噪音”，降低模型的推理精度。

### 📊 三种核心分块技术对比

| 分块方式 | 原理 | 优势 | 缺点 |
| :--- | :--- | :--- | :--- |
| **Fixed-size (固定尺寸)** | 使用固定字符长度（如 200 字）强行切分。 | 极低计算开销，实现极其简单。 | 可能在句子甚至词中间切断，导致语义丢失。 |
| **Recursive (递归切分)** | 按照的分隔符层级（如段落->句号->逗号）递归寻找切分点。 | 尽可能保持句子完整，语义连贯性好。 | 仍然是基于规则，无法感知语义的突变。 |
| **Semantic (语义切分)** | 利用模型（如 Embedding/LLM）检测语义波动，在意思发生明显变化处切分。 | 召回精度最高，能完美保留“语义块”。 | 计算开销大，切分速度慢。 |

---

In [ ]:
raw_doc = """大模型时代架构演进：从 NLP 到 RAG。
RAG 系统的核心在于检索。第一阶段是 Chunking。
我们将文本切分为若干个语义单元。每个单元称为一个 Chunk。
高效的 Chunking 能够提升检索的 Top-K 召回率。
此外，重排序（Rerank）也是必不可少的环节。
最后，提示词设计（Prompt Design）决定了生成的最终质量。"""

def fixed_size_chunking(text, size=20, overlap=5):
    """
    实现固定窗口切分。
    overlap 参数的物理意义：让相邻 Chunk 之间保留一部分重复内容，防止核心意思刚好被“切在缝隙里”。
    """
    chunks_list = []
    for i in range(0, len(text), size - overlap):
        chunks_list.append(text[i:i + size])
    return chunks_list

fixed_chunks = fixed_size_chunking(raw_doc)
print("🔴 Fixed-size Chunks (size=20, overlap=5):")
for i, c in enumerate(fixed_chunks):
    print(f"[{i}] {c.strip()}")

In [ ]:
def recursive_split_logic(text, separators=["\n", "。", "，"], max_size=30):
    """递归切分演示。它通过分隔符的层级降级来寻找最合适的语义断点。"""
    if len(text) <= max_size: return [text]
    for sep in separators:
        if sep in text:
            sub_texts = text.split(sep)
            res = []
            for st in sub_texts:
                res.extend(recursive_split_logic(st, separators[separators.index(sep)+1:], max_size))
            return [r for r in res if r]
    return [text]

recursive_chunks = recursive_split_logic(raw_doc)
print("🔵 Recursive Chunks (max_size=30):")
for i, c in enumerate(recursive_chunks):
    print(f"[{i}] {c.strip()}")

### 🟢 进阶演示：语义切分 (Semantic Chunking) 的底层逻辑

传统的切分（如 Fixed-size 或 Recursive）本质上是**基于规则**的。而语义切分是**基于模型理解**的。其工作流程如下：
1. **句子化**：将原始文档拆分为独立的句子。
2. **向量化**：利用 Embedding 模型获取每个句子的数学向量。
3. **差异检测**：计算相邻句子之间的**余弦相似度**。相似度越高，说明两句话在聊同一个话题。
4. **触发断点**：预设一个“相似度阈值”（如 0.8）。当相似度低于该值，模型认为语义发生了“突变”，从而在此处切开。

这种方式能确保每个 Chunk 内部的语义高度自治，是目前 RAG 检索精度最高的手法。

In [ ]:
def semantic_chunking_demo(text, model, tokenizer, threshold=0.7):
    """
    简易语义分块演示：计算相邻句子的相似度。
    如果相似度低于阈值，则认为语义发生突变，进行切分。
    """
    sentences = text.split("。")
    sentences = [s.strip() + "。" for s in sentences if s.strip()]
    
    if len(sentences) <= 1: return sentences
    
    final_chunks = []
    current_chunk = sentences[0]
    
    for i in range(1, len(sentences)):
        # 获取相邻两句的嵌入向量
        embs = get_embeddings([sentences[i-1], sentences[i]], model, tokenizer)
        import torch
        # 计算余弦相似度
        sim = torch.cosine_similarity(embs[0:1], embs[1:2]).item()
        
        if sim < threshold:
            final_chunks.append(current_chunk)
            current_chunk = sentences[i]
        else:
            current_chunk += sentences[i]
            
    final_chunks.append(current_chunk)
    return final_chunks

# 注意：此演示为了教学效果，将阈值设得较高以观察切分效果
semantic_chunks = semantic_chunking_demo(raw_doc, emb, emb_tok, threshold=0.8)
print("🟢 Semantic / LLM Chunks (Sim-Threshold=0.8):")
for i, c in enumerate(semantic_chunks):
    print(f"[{i}] {c.strip()}")

# 为了后续环节统一，我们将最终使用的 chunks 设为 recursive 版本
chunks = recursive_chunks

## 🔍 3. 第二支柱：相似度计算 (Vector Retrieval)

### 🔎 为什么要计算相似度？
在 RAG 中，我们不再使用“关键词匹配”，而是计算。这允许模型识别即使措辞完全不同但语义接近的内容（如“西红柿”与“番茄”）。

### 💡 深度热点答疑
1. **Chunk 长度不一，向量长度一样吗？**
   - **是一样的**。Embedding 模型通过 **Pooling (池化)** 技术，无论输入是 10 个词还是 500 个词，最终都会将其压缩到固定的维度（如 384 或 512 维）。
2. **为什么 Query Shape 是 [1, D]？**
   - 处理的是**单个查询**，所以 Batch Size 是 1；经过 Embedding 后，它变成了一个包含 D 个特征数值的向量。所以形状是 `[1, 特征维度]`。
3. **此 Embedding 与 Transformer 内部的 Embedding 一样吗？**
   - **结构层面上是一样的**，都是巨大的数值权重矩阵。但两者的**训练任务本质不同**：
     - **RAG Embedding**: 专门训练来识别“相似度”（Contrastive Learning）。
     - **LLM Embedding**: 专门训练来预测“下一个词”。
     - **不能随意更换**：不同模型维数不同且语义分布完全脱钩。更换 Embedding 必须重做整个数据库索引。

In [ ]:
import torch.nn.functional as F
import torch

def get_embeddings(texts, model, tokenizer):
    """将文本转化为固定维度的向量"""
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to("cpu")
    with torch.no_grad():
        outputs = model(**inputs)
        # 步骤 1: 取 [CLS] 位向量作为整句表示
        embeddings = outputs.last_hidden_state[:, 0, :]
        # 步骤 2: 归一化 (L2 Normalize)
        embeddings = F.normalize(embeddings, p=2, dim=1)
    return embeddings

# 1. 定义查询 (Query)
query = "RAG 的核心环节有哪些？"
query_emb = get_embeddings([query], emb, emb_tok)

# 2. 处理知识库片段 (Chunks)
chunk_embs = get_embeddings(chunks, emb, emb_tok)

print("🔍 检索环节：从人类语言到 Tensor 矩阵的映射")
print(f"   - 【问题输入】: \"{query}\"")
print(f"   - 【Query Tensor】: {query_emb.shape}  -> [1个问题, 512维特征值]")

print(f"\n   - 【知识库输入】: 共计 {len(chunks)} 个分块 (Chunks)")
# 打印这 8 个分块的内容预览，让学生看清“矩阵的每一行”到底对应哪句话
for i, c in enumerate(chunks):
    print(f"     └─ Chunk {i}: \"{c.strip()}\"")

print(f"\n   - 【Chunks Matrix Shape】: {chunk_embs.shape}  -> [{len(chunks)}个片段, 512维特征值]")

# 3. 执行向量检索 (相似度计算)
similarities = F.cosine_similarity(query_emb, chunk_embs, dim=1)

# 4. 提取结果
score, idx = torch.topk(similarities, k=1)
print(f"\n🏆 检索结论：相似度得分最高的是第 {idx.item()} 个片段 [相似度得分: {score.item():.4f}]")
print(f"   匹配内容: {chunks[idx].strip()}")

## ⚖️ 4. 第三支柱：重排序 (Rerank)

### 🔎 为什么要起名叫 “Cross-encoder”？
Embedding 属于 **Bi-encoder (双编码器)**：Query 和 Chunk 彼此不见面，检索时只是算点积，速度快但丢失了精细的互注意力。

**Cross-encoder** 正如其名，它在推理时将 `[Query]` 和 `[Chunk]` 拼接成一条。这意味着 Query 的每个词都能与 Chunk 的每个词发生“交叉关注” (Cross-Attention)，从而输出一个极其精准的“相关度分数”。

**输入输出定义：**
- **输入**: `[CLS] 问题文本 [SEP] 文档文本 [EOS]`
- **输出**: 该文档与问题相关的。 (单一分值，标量)

In [ ]:
def rerank_example(query, docs, model, tokenizer):
    """
    实现重排核心逻辑。
    注意关键参数：pairs 构造。我们将问题和多个文档进行配对打分。
    """
    pairs = [[query, doc] for doc in docs]
    inputs = tokenizer(pairs, padding=True, truncation=True, return_tensors="pt").to("cpu")
    
    with torch.no_grad():
        # Reranker 输出的是 logits (未归一化的原始分值)
        logits = model(**inputs).logits.view(-1)
        # 映射到 0-1 范围方便阅读观察
        rerank_scores = torch.sigmoid(logits)
    
    return rerank_scores

test_chunks = [chunks[0], chunks[1]]
scores = rerank_example(query, test_chunks, rerank, rerank_tok)

print(f"🧐 重排效果直观对比演示：")
print(f"👉 Query (查询问题): \"{query}\"")
print("-" * 50)

for i, s in enumerate(scores):
    print(f"🔹 Chunk {i} 原始文本: \"{test_chunks[i].strip()}\"")
    print(f"   ∟ 🚀 Cross-encoder 精排得分: {s:.4f}")
    print("-" * 50)

## ✍️ 5. 第四支柱：提示词工程 (Prompt Engineering Standard)

### 🔎 为什么采用 System + User 架构，而不是显式写在一个 Prompt 里？
在工程实践中，为了代码的清晰和模型的鲁棒性，我们使用角色分担机制：
- **System Prompt (全局契约)**: 告知模型它是谁、遵循什么死规矩（如拒答原则）。这部分是**静态**的。
- **User Prompt (动态上下文)**: 注入检索到的 `Context` 和 `Question`。这部分是。 

对于 Qwen/GPT 等 Chat 架构，这种分离能显著降低模型对指令的干扰，让模型更清晰地分辨什么是“资料”，什么是“问题”。

In [ ]:
def chat_with_qwen_rag(query, kb_chunks, kb_embs, retrieve_k=5, rerank_k=3):
    """
    工业级全链路 RAG 引擎：向量粗排 -> Cross-encoder 精排 -> Prompt 生成
    """
    # [1] 粗排阶段 (Coarse Retrieval)
    q_emb = get_embeddings([query], emb, emb_tok)
    similarities = F.cosine_similarity(q_emb, kb_embs, dim=1)
    _, coarse_indices = torch.topk(similarities, k=retrieve_k)
    coarse_texts = [kb_chunks[i.item()] for i in coarse_indices]
    
    # [2] 精排阶段 (Re-ranking)
    # 利用 Cross-encoder 对粗排结果进行更精细的互注意力打分
    rerank_scores = rerank_example(query, coarse_texts, rerank, rerank_tok)
    
    # 构造精排结果集并按得分降序重排
    ranked_results = []
    for i, score in enumerate(rerank_scores):
        ranked_results.append({
            "text": coarse_texts[i],
            "score": score.item(),
            "original_idx": coarse_indices[i].item()
        })
    ranked_results.sort(key=lambda x: x['score'], reverse=True)
    
    # [3] 构造 Prompt (取经排后的前 rerank_k 个资料)
    final_contexts = [r['text'] for r in ranked_results[:rerank_k]]
    system_p = "你是一个严谨的问答助手。请阅读参考资料回答问题。\n如果问题与资料的主题无关，则回答\"主题无关\"。\n如果资料没提到与问题相关的内容，请告知我\"资料中未提及\"。"
    context_block = "\n".join([f"[资料 {i+1}]: {t.strip()}" for i, t in enumerate(final_contexts)])
    user_p = f"参考资料：\n{context_block}\n\n我的问题：{query}"
    
    messages = [{"role": "system", "content": system_p}, {"role": "user", "content": user_p}]
    text = llm_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = llm_tok([text], return_tensors="pt").to("cpu")
    
    with torch.no_grad():
        generated_ids = llm.generate(**model_inputs, max_new_tokens=100, do_sample=False, pad_token_id=llm_tok.pad_token_id)
        response_ids = generated_ids[0][len(model_inputs.input_ids[0]):]
        answer = llm_tok.decode(response_ids, skip_special_tokens=True).strip()
        
    return answer.split("\n")[0], ranked_results

# 演示全链路运行过程
answer, ranked_details = chat_with_qwen_rag(query, chunks, chunk_embs)

print(f"👉 Query (查询问题): \"{query}\"")
print(f"⚖️ Rerank 精排报告 (候选 Top-5 已按重排得分降序排序)：")
for i, res in enumerate(ranked_details):
    symbol = "✅" if i == 0 else "🔹"
    print(f"   {symbol} Rank {i+1} [得分: {res['score']:.4f}] 内容: \"{res['text'].strip()}\"")

print(f"\n🤖 最终生成的 RAG 回复：\n   {answer}")

## 🧪 6. 鲁棒性验证：RAG 工作流演示

我们测试两个关键 Case：
1. **知识召回**：证明 RAG 能够扩充模型的实时知识。
2. **边界安全**：证明 RAG 能够抑制模型乱答无关问题。

In [ ]:
test_queries = [
    "RAG 的第一阶段是什么？",
    "今天是星期几？"
]

for q in test_queries:
    # 此时 RAG 逻辑已内化粗排与精排两个环节
    ans, _ = chat_with_qwen_rag(q, chunks, chunk_embs)
    print(f"【问题】: {q}")
    print(f"【回答】: {ans}\n" + "-"*40)

## 📈 7. 本课知识复盘 (Summary)

通过本实验，你应该掌握以下 RAG 核心概念：
1. **Chunking**: 固定尺寸 vs 递归切分的折衷（速度与语义的平衡）。
2. **Embedding**: 为什么模型不论输入多长，最后都是固定维度的特征向量。
3. **Rerank**: Cross-encoder 利用双端注意力弥补了向量检索“粗略”的不足。
4. **Prompt Standard**: 设计工业级 RAG 提示词应遵循角色分离原则，以增强稳定性。